In [2]:


import os
import sys
import wave
import struct
import math



def transcribe_with_speech_recognition(audio_path: str,
                                        engine: str = "google") -> str:

    try:
        import speech_recognition as sr
    except ImportError:
        return "❌  Please run: pip install SpeechRecognition"

    recognizer = sr.Recognizer()

    try:
        with sr.AudioFile(audio_path) as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio_data = recognizer.record(source)
    except FileNotFoundError:
        return f"❌  File not found: {audio_path}"
    except Exception as e:
        return f"❌  Could not read audio file: {e}"

    try:
        if engine == "google":
            text = recognizer.recognize_google(audio_data)
        elif engine == "sphinx":
            text = recognizer.recognize_sphinx(audio_data)
        else:
            return f"❌  Unknown engine: {engine}"
        return text
    except Exception as e:
        return f"❌  Recognition failed: {e}"


def transcribe_from_microphone(duration: int = 5,
                                engine: str = "google") -> str:

    try:
        import speech_recognition as sr
    except ImportError:
        return "❌  Please run: pip install SpeechRecognition pyaudio"

    recognizer = sr.Recognizer()

    print(f"🎙️  Listening for {duration} second(s) … speak now!")
    try:
        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio_data = recognizer.listen(source, timeout=duration,
                                           phrase_time_limit=duration)
    except Exception as e:
        return f"❌  Microphone error: {e}"

    print("⏳  Transcribing …")
    try:
        if engine == "google":
            return recognizer.recognize_google(audio_data)
        elif engine == "sphinx":
            return recognizer.recognize_sphinx(audio_data)
        else:
            return f"❌  Unknown engine: {engine}"
    except Exception as e:
        return f"❌  Recognition failed: {e}"




def _create_demo_wav(path: str = "demo_audio.wav",
                     duration: float = 2.0,
                     frequency: float = 440.0,
                     sample_rate: int = 16000):
    """Write a pure-tone WAV so the demo runs without a real audio file."""
    num_samples = int(sample_rate * duration)
    with wave.open(path, "w") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)          # 16-bit
        wf.setframerate(sample_rate)
        for i in range(num_samples):
            value = int(32767 * math.sin(
                2 * math.pi * frequency * i / sample_rate
            ))
            wf.writeframes(struct.pack("<h", value))
    return path



def run_demo():
    print("=" * 65)

    print("=" * 65)

    # ── Demo 1: transcribe a WAV file ─────────────────────────────
    demo_wav = "demo_audio.wav"
    _create_demo_wav(demo_wav)
    print(f"\n📁  Transcribing demo WAV file: {demo_wav}")
    result = transcribe_with_speech_recognition(demo_wav, engine="google")
    print(f"   Transcript: {result}")
    os.remove(demo_wav)

    # ── Demo 2: transcribe a user-supplied file ───────────────────
    print("\n" + "-" * 65)
    print("📂  OPTION 1 – Transcribe your own audio file")
    audio_file = input("   Enter path to .wav / .flac / .aiff file "
                       "(or press Enter to skip): ").strip()
    if audio_file:
        engine_choice = input("   Engine [google / sphinx, default=google]: ").strip() or "google"
        print(f"\n⏳  Transcribing '{audio_file}' with {engine_choice} engine …")
        transcript = transcribe_with_speech_recognition(audio_file, engine=engine_choice)
        print(f"\n✅  Transcript:\n   {transcript}")

    # ── Demo 3: microphone ────────────────────────────────────────
    print("\n" + "-" * 65)
    print("🎙️  OPTION 2 – Live microphone transcription")
    use_mic = input("   Record from microphone? [y/N]: ").strip().lower()
    if use_mic == "y":
        try:
            secs = int(input("   Duration in seconds [default 5]: ") or 5)
        except ValueError:
            secs = 5
        transcript = transcribe_from_microphone(duration=secs, engine="google")
        print(f"\n✅  Transcript:\n   {transcript}")

    print("\n" + "=" * 65)

    print("=" * 65)


if __name__ == "__main__":
    run_demo()


📁  Transcribing demo WAV file: demo_audio.wav
   Transcript: ❌  Please run: pip install SpeechRecognition

-----------------------------------------------------------------
📂  OPTION 1 – Transcribe your own audio file
   Enter path to .wav / .flac / .aiff file (or press Enter to skip): "C:\Users\itsme\Downloads\audio.ogg"
   Engine [google / sphinx, default=google]: google

⏳  Transcribing '"C:\Users\itsme\Downloads\audio.ogg"' with google engine …

✅  Transcript:
   ❌  Please run: pip install SpeechRecognition

-----------------------------------------------------------------
🎙️  OPTION 2 – Live microphone transcription
   Record from microphone? [y/N]: N

